# FINAL LITHOLOGY MODEL

Settings chosen in `feature_testing.ipynb`: cleaned data, **4 m rolling median** of the five curves,
RobustScaler, **KMeans k = 3**. This notebook runs them, names the clusters from their readings, and saves
`force2020_lithology.csv` — the labelled file the predictor notebook trains on.

The lithology names are interpretations of log behaviour. This dataset has no labels, so they cannot be
checked for accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("force2020_cleaned.csv")
curves = ["RHOB", "GR", "NPHI", "PEF", "DTC"]
units = {"RHOB": "g/cc", "GR": "API", "NPHI": "v/v", "PEF": "b/e", "DTC": "us/ft"}
step = df["DEPTH_MD"].diff().median()
window = int(round(4 / step)) | 1

print("Rows:", len(df), "| step:", round(step, 3), "m | 4 m window =", window, "rows")

In [ ]:
features = pd.DataFrame({f"{col}_4m": df[col].rolling(window, center=True, min_periods=window // 2 + 1).median()
                         for col in curves}, index=df.index)

model_rows = features.notna().all(axis=1)
X = pd.DataFrame(RobustScaler().fit_transform(features[model_rows]),
                 index=features.index[model_rows], columns=features.columns)

print("Depths clustered:", len(X), "| range:",
      round(df.loc[model_rows, "DEPTH_MD"].min(), 1), "-", round(df.loc[model_rows, "DEPTH_MD"].max(), 1), "m")

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
df.loc[model_rows, "cluster"] = kmeans.labels_

print("silhouette:", round(silhouette_score(X, kmeans.labels_, sample_size=5000, random_state=42), 3))

profile = df.loc[model_rows].groupby("cluster")[curves].median().round(2)
profile["rows"] = df.loc[model_rows, "cluster"].value_counts().sort_index()
profile["top (m)"] = df.loc[model_rows].groupby("cluster")["DEPTH_MD"].min().round(1)
profile["bottom (m)"] = df.loc[model_rows].groupby("cluster")["DEPTH_MD"].max().round(1)
profile

In [ ]:
# name the clusters from their readings, not from their numbers (KMeans numbering is arbitrary)
carbonate = profile["GR"].idxmin()                                  # least clay
soft_shale = profile.drop(index=carbonate)["DTC"].idxmax()          # slowest = least compacted
compacted = [c for c in profile.index if c not in (carbonate, soft_shale)][0]

names = {carbonate: "Limestone / chalk", soft_shale: "Soft shale", compacted: "Compacted shale"}
litho_colors = {"Soft shale": "#8c6d46", "Limestone / chalk": "#2a78d6", "Compacted shale": "#3a9e5f"}

df["lithology"] = df["cluster"].map(names)
print(names)
df["lithology"].value_counts()

In [ ]:
lith = df.loc[model_rows, ["DEPTH_MD", "lithology"]]
block = (lith["lithology"] != lith["lithology"].shift()).cumsum()

layers = lith.groupby(block).agg(lithology=("lithology", "first"), top=("DEPTH_MD", "min"),
                                 bottom=("DEPTH_MD", "max"), rows=("DEPTH_MD", "size"))
layers["thickness (m)"] = (layers["bottom"] - layers["top"] + step).round(1)

print("Layers found:", len(layers), "| at least 5 m thick:", int((layers["thickness (m)"] >= 5).sum()))
layers[layers["thickness (m)"] >= 5].reset_index(drop=True).round(1)

In [ ]:
track_range = {"RHOB": (1.4, 3.0), "GR": (0, 250), "NPHI": (0, 0.8), "PEF": (0, 10), "DTC": (50, 180)}
rows = df[model_rows]

fig, axes = plt.subplots(1, len(curves) + 1, figsize=(15, 13), sharey=True)
for ax, col in zip(axes, curves):
    ax.plot(features.loc[model_rows, f"{col}_4m"], rows["DEPTH_MD"], color="#555555", linewidth=0.6)
    ax.set_xlim(track_range[col])
    ax.set_title(f"{col} ({units[col]})")
    ax.grid(color="#e1e0d9", linewidth=0.6)
    ax.set_axisbelow(True)

strip = axes[-1]
for name, color in litho_colors.items():
    strip.fill_betweenx(rows["DEPTH_MD"], 0, 1, where=rows["lithology"] == name, color=color)
strip.set_xticks([])
strip.set_title("Estimated lithology")
strip.legend(handles=[Patch(facecolor=color, label=name) for name, color in litho_colors.items()],
             loc="lower left", bbox_to_anchor=(1, 0), frameon=False)

axes[0].set_ylabel("Depth (m)")
axes[0].invert_yaxis()
fig.suptitle("Estimated lithology from 3-cluster KMeans on 4 m median curves", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (x, y) in zip(axes, [("NPHI", "RHOB"), ("NPHI", "PEF")]):
    for name, color in litho_colors.items():
        pick = rows["lithology"] == name
        ax.scatter(rows.loc[pick, x], rows.loc[pick, y], s=4, alpha=0.5, color=color, label=name)
    ax.set_xlabel(f"{x} ({units[x]})")
    ax.set_ylabel(f"{y} ({units[y]})")
    ax.set_title(f"{y} vs {x}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, markerscale=3)
fig.suptitle("Estimated lithology on the crossplots (raw readings)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
output = df[["DEPTH_MD"] + curves + ["cluster", "lithology"]].copy()
output.to_csv("force2020_lithology.csv", index=False)

check = pd.read_csv("force2020_lithology.csv")
print("Saved:", check.shape[0], "rows,", check.shape[1], "columns")
print("Labelled depths:", int(check["lithology"].notna().sum()))
check.dropna(subset=["lithology"]).head(3)

### Result

**Three rock units, 1138.7–2993.9 m** (12,104 depths, silhouette 0.576):

| Estimated rock | RHOB | GR | NPHI | PEF | DTC | Depths | Main interval |
|---|---|---|---|---|---|---|---|
| Soft shale | 2.01 | 65 | 0.50 | 2.8 | 145 | 8,184 | 1138.7–2394.8 m |
| Limestone / chalk | 2.54 | 18 | 0.18 | 4.7 | 71 | 2,328 | 2439.5–2732.3 m |
| Compacted shale | 2.48 | 96 | 0.31 | 4.4 | 88 | 1,592 | 2790.2–2993.9 m |

**Why these names:** the carbonate has almost no clay (GR 18), is dense and fast — limestone or chalk. Both
shales are clay-rich, and the deep one is dense and fast because burial has compacted it, while the upper one
is light, porous and slow.

**How reliable:** the boundaries near **2395 m** and **2741 m** were found by KMeans, Gaussian Mixture,
HDBSCAN and Agglomerative alike, before and after cleaning, at every window size from 2 to 8 m. The thin
alternating layers between 2233–2440 m and 2732–2790 m are transition zones where the two rock types
interfinger, not separate units.

**Limits:** no lithology labels exist in this dataset, so the names are interpretation; coal, dolomite and
sandstone cannot be told apart from these five curves alone; nothing above 1138.7 m or below 2993.9 m can be
classified, because NPHI and PEF were not recorded there.

**Output:** `force2020_lithology.csv` — depth, five cleaned curves, cluster number and lithology name, with
blanks outside the logged interval. This is the training data for `lithology_predictor.ipynb`.